In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, accuracy_score
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import tensorflow as tf

print("Импорты выполнены.")
# Отключаем лишние логи TensorFlow
tf.get_logger().setLevel('ERROR')

Импорты выполнены.


In [2]:
def tpr_at_1pct_fpr(y_true, y_pred_proba):
    """
    Доля правильно обнаруженных атак при 1% ложных срабатываний.
    """
    sorted_indices = np.argsort(y_pred_proba)[::-1]
    sorted_true = y_true[sorted_indices]
    
    total_neg = np.sum(y_true == 0)
    if total_neg == 0:
        return 0.0
    
    target_fp = int(0.01 * total_neg)
    fp = 0
    tp = 0
    for label in sorted_true:
        if label == 0:
            fp += 1
        else:
            tp += 1
        if fp >= target_fp:
            break
    
    total_pos = np.sum(y_true == 1)
    if total_pos == 0:
        return 0.0
    return tp / total_pos

In [ ]:
df = pd.read_csv("../data/processed/balanced_unified_dataset.csv")  # путь к общему файлу
print(f"Всего строк: {len(df)}")
print("Уникальные источники:", df['source'].unique())

Всего строк: 70032
Уникальные источники: ['fwaf' 'csic2010' 'vulnxss' 'kaggle' 'httpparams' 'payloadbox']


In [4]:
MAX_LEN = 200          # максимальная длина последовательности (урезаем или дополняем)
VOCAB_SIZE = 10000     # размер словаря (берём топ-10000 слов)
EMBEDDING_DIM = 64     # размерность эмбеддингов
LSTM_UNITS = 64        # количество LSTM-нейронов
DROPOUT = 0.2          # dropout внутри LSTM
RECURRENT_DROPOUT = 0.2
DENSE_UNITS = 32
DENSE_DROPOUT = 0.3
EPOCHS = 3             # для скорости, можно увеличить до 5-10 для лучших результатов
BATCH_SIZE = 64

In [5]:
source_names = ['fwaf', 'csic2010', 'kaggle', 'vulnxss', 'httpparams']
results = {}

for test_source in source_names:
    print(f"\n===== Тестируем на {test_source} =====")
    
    # Разделение на train/test по источнику
    test_mask = (df['source'] == test_source)
    train_mask = ~test_mask
    
    texts_train = df.loc[train_mask, 'text'].values
    texts_test = df.loc[test_mask, 'text'].values
    y_train = df.loc[train_mask, 'label'].values
    y_test = df.loc[test_mask, 'label'].values
    
    print(f"  Обучающих: {len(texts_train)}, тестовых: {len(texts_test)}")
    
    # Токенизация на обучающих данных
    tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token='<OOV>')
    tokenizer.fit_on_texts(texts_train)
    
    # Преобразование текстов в последовательности индексов
    X_train_seq = tokenizer.texts_to_sequences(texts_train)
    X_test_seq = tokenizer.texts_to_sequences(texts_test)
    
    # Паддинг (выравнивание длины) до MAX_LEN
    X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
    X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')
    
    # Строим модель
    model = Sequential([
        Embedding(VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_LEN),
        Bidirectional(LSTM(LSTM_UNITS, dropout=DROPOUT, recurrent_dropout=RECURRENT_DROPOUT)),
        Dense(DENSE_UNITS, activation='relu'),
        Dropout(DENSE_DROPOUT),
        Dense(1, activation='sigmoid')
    ])
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    
    # Early stopping для предотвращения переобучения
    early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)
    
    # Обучение
    history = model.fit(
        X_train_pad, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=0.2,
        callbacks=[early_stop],
        verbose=0  # убираем вывод прогресса, чтобы не засорять консоль
    )
    
    # Предсказания вероятностей
    y_proba = model.predict(X_test_pad, verbose=0).flatten()
    y_pred = (y_proba >= 0.5).astype(int)
    
    # Метрики
    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    tpr = tpr_at_1pct_fpr(y_test, y_proba)
    
    results[test_source] = {
        'roc_auc': roc_auc,
        'pr_auc': pr_auc,
        'f1': f1,
        'accuracy': acc,
        'tpr@1%fpr': tpr
    }
    
    print(f"  ROC-AUC: {roc_auc:.4f}, PR-AUC: {pr_auc:.4f}, F1: {f1:.4f}, Accuracy: {acc:.4f}, TPR@1%FPR: {tpr:.4f}")

print("\nЦикл завершён.")


===== Тестируем на fwaf =====
  Обучающих: 35907, тестовых: 34125


d:\учёба\SSRW\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:1046: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


  ROC-AUC: nan, PR-AUC: 0.0000, F1: 0.0000, Accuracy: 0.7989, TPR@1%FPR: 0.0000

===== Тестируем на csic2010 =====
  Обучающих: 44873, тестовых: 25159


d:\учёба\SSRW\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


  ROC-AUC: nan, PR-AUC: 1.0000, F1: 0.9318, Accuracy: 0.8723, TPR@1%FPR: 0.0000

===== Тестируем на kaggle =====
  Обучающих: 62689, тестовых: 7343


...

  ROC-AUC: 0.7540, PR-AUC: 0.9954, F1: 0.9910, Accuracy: 0.9823, TPR@1%FPR: 0.0001

===== Тестируем на vulnxss =====
  Обучающих: 68504, тестовых: 1528


d:\учёба\SSRW\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
d:\учёба\SSRW\.venv\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


  ROC-AUC: nan, PR-AUC: 1.0000, F1: 0.9563, Accuracy: 0.9162, TPR@1%FPR: 0.0000

===== Тестируем на httpparams =====
  Обучающих: 69005, тестовых: 1027


d:\учёба\SSRW\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


  ROC-AUC: 0.9976, PR-AUC: 0.9923, F1: 0.9405, Accuracy: 0.9718, TPR@1%FPR: 0.9056

Цикл завершён.


In [6]:
result_df = pd.DataFrame(results).T
print("\n=== Итоговая таблица LODO (LSTM) ===")
print(result_df.round(4))

result_df.to_csv("lodo_lstm.csv", index=True)
print("Результаты сохранены в lodo_lstm.csv")


=== Итоговая таблица LODO (LSTM) ===
            roc_auc  pr_auc      f1  accuracy  tpr@1%fpr
fwaf            NaN  0.0000  0.0000    0.7989     0.0000
csic2010        NaN  1.0000  0.9318    0.8723     0.0000
kaggle       0.7540  0.9954  0.9910    0.9823     0.0001
vulnxss         NaN  1.0000  0.9563    0.9162     0.0000
httpparams   0.9976  0.9923  0.9405    0.9718     0.9056
Результаты сохранены в lodo_lstm.csv
